# GRAFT Phase 2: Q&A Dataset Generation
This phase demonstrates how the hierarchical GraphRAG summaries are used to build a robust QA dataset, complete with oracle graphs and distractor communities.

In [ ]:
import sys
import os
import json

sys.path.append(os.path.join("..", "src"))
from graph_indexer import GraphIndexer
from raft_dataset_builder import RAFTDatasetBuilder

indexer = GraphIndexer()
indexer.load_index("../data/graph_index")
print(f"Loaded {len(indexer.community_summaries)} communities.")

In [ ]:
builder = RAFTDatasetBuilder()

# 1. Question Generation
questions = builder.generate_questions(indexer.community_summaries, 1)
print(f"Sample Question: {questions[0].text} (Type: {questions[0].type})")

In [ ]:
# 2. Context Assembly & Distractor Injection
sample_q = questions[0]
oracle_ctx = builder.build_oracle_context(sample_q, indexer.community_summaries, indexer.graph)
aug_ctx = builder.inject_distractor_documents(oracle_ctx, indexer.community_summaries, 2)

print("Augmented Context IDs:", [c.source for c in aug_ctx])
print("Oracle count:", sum(1 for c in aug_ctx if c.is_oracle))

In [ ]:
# 3. Gold Standard Generation (CoT)
cot = builder.generate_cot_answer(sample_q, oracle_ctx)

print("REASONING:", cot.reasoning)
print("ANSWER:", cot.final_answer)
print("CITATIONS:", cot.citations)

In [ ]:
# 4. Save Final JSONL Sample
sample = builder.build_training_sample(sample_q, aug_ctx, cot)
builder.export_dataset([sample], "../data/processed", "jsonl")

print("Created dataset sample:")
print(json.dumps(sample.__dict__, indent=2))